# Movie Flop Predictor — Aplicație Web

## Scopul aplicației

În această etapă construim o aplicație web simplă care folosește modelul final de tip **Gradient Boosting optimizat** pentru a estima performanța financiară relativă a unui film.

Utilizatorul informații despre un film, precum bugetul, rating-ul, popularitatea, durata și luna lansării, apoi estimează valoarea **Disappointment Index**.

Pe baza acestei valori, aplicația clasifică filmul în una dintre următoarele categorii:

- risc ridicat de flop;
- performanță apropiată de așteptări;
- succes peste așteptări.

Această aplicație are rol demonstrativ și arată cum poate fi utilizat modelul construit anterior într-un context practic.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

import joblib
import warnings
warnings.filterwarnings('ignore')

## 1. Încărcarea datelor

Începem prin încărcarea datasetului curățat, obținut în etapa de preprocessing. Acesta conține doar filmele cu date financiare valide și variabilele necesare pentru modelare.

In [2]:
df = pd.read_csv('../data/movies_clean.csv')

print(f"Dataset încărcat: {df.shape[0]} filme")
df.head()

Dataset încărcat: 866 filme


,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,...,vote_average,vote_count,budget,revenue,runtime,expected_revenue,disappointment_index,roi,release_year,release_month
0,False,/2I1OFQJ0L9T0dpU6FobKFWV2PxX.jpg,"[878, 12]",687163,Project Hail Mary,en,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,417.1089,/yihdXomYb5kTeSivtFndMy5iDmf.jpg,...,8.602,3681,200000000,668516856,157,500000000.0,1.337034,2.342584,2026,3
1,False,/9Z2uDYXqJrlmePznQQJhL6d92Rq.jpg,"[10751, 35, 12, 14, 16]",1226863,The Super Mario Galaxy Movie,en,The Super Mario Galaxy Movie,Having thwarted Bowser's previous plot to marr...,332.9093,/eJGWx219ZcEMVQJhAgMiqo8tYY.jpg,...,7.159,1083,110000000,967144200,98,275000000.0,3.516888,7.792220,2026,4
2,False,/wMrV8SLne1jHLeYS0lLrA1Tf86P.jpg,"[27, 9648]",1304313,Lee Cronin's The Mummy,en,Lee Cronin's The Mummy,The young daughter of a journalist disappears ...,303.3424,/1q308iixueCU4pFtSYugNOevtNx.jpg,...,7.706,484,22000000,89191878,133,55000000.0,1.621671,3.054176,2026,4
3,False,/gkh6Nt8DtY1XT4gQsyFq9XAVJlJ.jpg,"[18, 35]",350,The Devil Wears Prada,en,The Devil Wears Prada,A young woman from the Midwest gets more than ...,257.6913,/8912AsVuS7Sj915apArUFbv6F9L.jpg,...,7.415,13487,35000000,326588371,109,87500000.0,3.732439,8.331096,2006,6
4,False,/6ELCZlTA5lGUops70hKdB83WJxH.jpg,"[28, 14, 12]",460465,Mortal Kombat,en,Mortal Kombat,"Washed-up MMA fighter Cole Young, unaware of h...",159.5253,/ybrX94xQm8lXYpZAPRmwD9iIbWP.jpg,...,7.025,6566,20000000,84426031,110,50000000.0,1.688521,3.221302,2021,4


## 2. Construirea variabilelor suplimentare

Pentru aplicația web folosim atât variabilele inițiale, cât și variabile construite prin feature engineering.

Am adăugat:

- `log_budget` — transformare logaritmică a bugetului;
- `log_vote_count` — transformare logaritmică a numărului de voturi;
- `log_popularity` — transformare logaritmică a popularității;
- `movie_age` — vârsta filmului;
- `is_summer_release` — dacă filmul a fost lansat vara;
- `is_december_release` — dacă filmul a fost lansat în decembrie;
- `is_long_movie` — dacă filmul are durata mai mare de 120 minute.

Aceste variabile ajută modelul să surprindă mai bine relațiile neliniare dintre caracteristicile filmului și performanța sa financiară.

In [3]:
df['log_budget'] = np.log1p(df['budget'])
df['log_vote_count'] = np.log1p(df['vote_count'])
df['log_popularity'] = np.log1p(df['popularity'])

df['movie_age'] = 2026 - df['release_year']
df['is_summer_release'] = df['release_month'].isin([6, 7, 8]).astype(int)
df['is_december_release'] = (df['release_month'] == 12).astype(int)
df['is_long_movie'] = (df['runtime'] > 120).astype(int)

df[[
    'log_budget', 'log_vote_count', 'log_popularity',
    'movie_age', 'is_summer_release',
    'is_december_release', 'is_long_movie'
]].head()

,log_budget,log_vote_count,log_popularity,movie_age,is_summer_release,is_december_release,is_long_movie
0,19.113828,8.211211,6.035742,0,0,0,1
1,18.515991,6.988413,5.810869,0,0,0,0
2,16.906553,6.184149,5.718153,0,0,0,1
3,17.370859,9.509556,5.555635,20,1,0,0
4,16.811243,8.789812,5.078452,5,0,0,0


## 3. Alegerea variabilelor pentru modelul aplicației

Pentru aplicație folosim setul de variabile selectate în etapa de feature selection. Acesta a avut cea mai bună performanță dintre variantele testate.

Variabilele folosite sunt:

- `movie_age`
- `log_budget`
- `budget`
- `release_year`
- `vote_count`
- `vote_average`
- `log_popularity`
- `log_vote_count`

In [4]:
features_app = [
    'movie_age',
    'log_budget',
    'budget',
    'release_year',
    'vote_count',
    'vote_average',
    'log_popularity',
    'log_vote_count'
]

X = df[features_app]
y = df['disappointment_index']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape[0]} filme")
print(f"Test: {X_test.shape[0]} filme")

Train: 692 filme
Test: 174 filme


## 4. Antrenarea modelului final

Pentru aplicația web folosim modelul **Gradient Boosting optimizat**, deoarece acesta a obținut cea mai bună performanță în etapa de modelare și hiperparametrizare.

Hiperparametrii utilizați sunt cei rezultați în urma procesului de Grid Search.

In [5]:
final_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=2,
    min_samples_leaf=1,
    subsample=0.8,
    random_state=42
)

final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)

r2_final = r2_score(y_test, y_pred)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred))

print("Model final pentru aplicație:")
print(f"R2 pe test: {r2_final:.4f}")
print(f"RMSE pe test: {rmse_final:.4f}")

Model final pentru aplicație:
R2 pe test: 0.5167
RMSE pe test: 5.6785


## 5. Salvarea modelului

Pentru ca modelul să poată fi folosit în aplicația web, îl salvăm într-un fișier `.pkl`. Salvăm și lista variabilelor folosite, astfel încât aplicația să trimită datele către model în aceeași ordine ca la antrenare.

In [6]:
joblib.dump(final_model, '../data/final_movie_flop_model.pkl')
joblib.dump(features_app, '../data/features_app.pkl')

print("Modelul și lista de variabile au fost salvate cu succes.")

Modelul și lista de variabile au fost salvate cu succes.


## 6. Test rapid de predicție

Înainte de a construi aplicația web, testăm modelul pe un exemplu fictiv de film. Astfel verificăm dacă modelul primește corect datele și returnează o valoare estimată pentru Disappointment Index.

In [7]:
example_movie = pd.DataFrame({
    'movie_age': [1],
    'log_budget': [np.log1p(100_000_000)],
    'budget': [100_000_000],
    'release_year': [2025],
    'vote_count': [5000],
    'vote_average': [7.2],
    'log_popularity': [np.log1p(80)],
    'log_vote_count': [np.log1p(5000)]
})

prediction = final_model.predict(example_movie)[0]

print(f"Disappointment Index estimat: {prediction:.2f}")

Disappointment Index estimat: 1.65


## 7. Interpretarea predicției

Pentru aplicația web, valoarea estimată a Disappointment Index este transformată într-un mesaj ușor de înțeles.

Folosim următoarea logică:

- dacă DI < 1 → filmul are risc ridicat de flop;
- dacă DI este între 1 și 3 → filmul are performanță apropiată de așteptări;
- dacă DI > 3 → filmul are potențial de succes peste așteptări.

Această interpretare permite utilizatorului să înțeleagă rapid rezultatul modelului.

In [8]:
def interpret_prediction(di):
    if di < 1:
        return "Risc ridicat de flop — filmul este estimat să performeze sub așteptări."
    elif di < 3:
        return "Performanță apropiată de așteptări — filmul poate recupera investiția."
    else:
        return "Succes peste așteptări — filmul are potențial de performanță financiară foarte bună."

print(interpret_prediction(prediction))

Performanță apropiată de așteptări — filmul poate recupera investiția.


## 8. Construirea aplicației web cu Streamlit

Aplicația va fi salvată într-un fișier separat numit `app.py`.

Aceasta va permite utilizatorului să introducă valorile principale ale unui film, iar modelul va estima automat Disappointment Index și categoria de risc.

In [9]:
%%writefile ../app.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib

# Încărcăm modelul și lista de variabile
model = joblib.load('data/final_movie_flop_model.pkl')
features_app = joblib.load('data/features_app.pkl')

st.set_page_config(
    page_title="Movie Flop Predictor",
    page_icon="🎬",
    layout="centered"
)

st.title("🎬 Movie Flop Predictor")
st.write("""
Această aplicație estimează performanța financiară relativă a unui film,
folosind modelul final Gradient Boosting optimizat.
""")

st.markdown("---")

st.subheader("Introduceți caracteristicile filmului")

budget = st.number_input(
    "Buget film ($)",
    min_value=100000,
    max_value=500000000,
    value=100000000,
    step=1000000
)

vote_average = st.slider(
    "Rating mediu",
    min_value=0.0,
    max_value=10.0,
    value=7.0,
    step=0.1
)

vote_count = st.number_input(
    "Număr voturi",
    min_value=0,
    max_value=50000,
    value=5000,
    step=100
)

popularity = st.number_input(
    "Popularitate",
    min_value=0.0,
    max_value=1000.0,
    value=80.0,
    step=1.0
)

runtime = st.number_input(
    "Durată film (minute)",
    min_value=30,
    max_value=300,
    value=120,
    step=5
)

release_year = st.number_input(
    "An lansare",
    min_value=1950,
    max_value=2026,
    value=2025,
    step=1
)

release_month = st.selectbox(
    "Luna lansării",
    options=list(range(1, 13)),
    index=5
)

# Construim variabilele folosite de model
movie_age = 2026 - release_year
log_budget = np.log1p(budget)
log_vote_count = np.log1p(vote_count)
log_popularity = np.log1p(popularity)

input_data = pd.DataFrame({
    'movie_age': [movie_age],
    'log_budget': [log_budget],
    'budget': [budget],
    'release_year': [release_year],
    'vote_count': [vote_count],
    'vote_average': [vote_average],
    'log_popularity': [log_popularity],
    'log_vote_count': [log_vote_count]
})

input_data = input_data[features_app]

def interpret_prediction(di):
    if di < 1:
        return "🔴 Risc ridicat de flop — filmul este estimat să performeze sub așteptări."
    elif di < 3:
        return "🟡 Performanță apropiată de așteptări — filmul poate recupera investiția."
    else:
        return "🟢 Succes peste așteptări — filmul are potențial financiar ridicat."

st.markdown("---")

if st.button("Estimează performanța filmului"):
    prediction = model.predict(input_data)[0]

    st.subheader("Rezultat predicție")
    st.metric("Disappointment Index estimat", f"{prediction:.2f}")

    interpretation = interpret_prediction(prediction)
    st.write(interpretation)

    st.markdown("""
    **Cum interpretăm rezultatul?**

    Disappointment Index compară performanța estimată a filmului cu venitul așteptat în funcție de buget.
    O valoare sub 1 indică o performanță sub așteptări, iar o valoare peste 1 indică faptul că filmul poate depăși pragul estimat.
    """)

Writing ../app.py


## 9. Rularea aplicației

Pentru a porni aplicația web, deschidem terminalul în folderul principal al proiectului și rulăm comanda:

```bash
python -m streamlit run app.py

Dacă nu avem pachetul streamlit întâi comanda: 
python -m pip install streamlit

Cu ctrl+c oprim aplicatia